In [1]:
# 1. Install txtify and LEANN
!pip install -q txtify leann





In [ ]:
#Do not use
# 2. Clean any partial clone and clone the official snap-stanford/memorilla repo
!rm -rf /content/memorilla
!git config --global http.version HTTP/1.1
!git clone https://github.com/snap-stanford/memorilla.git /content/memorilla

# Install Memorilla in editable mode
!pip install -q -e /content/memorilla

In [2]:
# 3. Download PDF directly from URL
import urllib.request

PDF_URL = "https://dairysciencepark.org/wp-content/uploads/2022/09/The-100-Ranking-of-Most-Influential-People-in-History-Michael-Hart-Citadel-Press-1992-PDFDrive-.pdf"  # <-- Replace with your PDF URL
PDF_FILENAME = "/content/LeanNInput.pdf"

req = urllib.request.Request(
    PDF_URL,
    headers={"User-Agent": "Mozilla/5.0"}
)
with urllib.request.urlopen(req) as response, open(PDF_FILENAME, "wb") as out_file:
    out_file.write(response.read())

print(f"Downloaded '{PDF_FILENAME}' from {PDF_URL}")




Downloaded '/content/LeanNInput.pdf' from https://dairysciencepark.org/wp-content/uploads/2022/09/The-100-Ranking-of-Most-Influential-People-in-History-Michael-Hart-Citadel-Press-1992-PDFDrive-.pdf


In [4]:
import gc
import subprocess
from pathlib import Path
import torch
from leann import LeannBuilder, LeannSearcher, LeannChat

# --- Step 1: Extract text using txtify ---
PDF_FILENAME="/content/LeanNInput.pdf"
output_dir = Path("/content")
output_dir.mkdir(parents=True, exist_ok=True)

subprocess.run(["txtify", PDF_FILENAME, "-o", str(output_dir)], check=True)



CompletedProcess(args=['txtify', '/content/LeanNInput.pdf', '-o', '/content'], returncode=0)

In [2]:
extracted_txt_path = "/content/LeanExtract.txt"

with open(extracted_txt_path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

print(f"[txtify] Extracted {len(lines)} lines.")



[txtify] Extracted 13164 lines.


In [5]:
# Do not use
# --- Step 2: Insert every line into LEANN ---
INDEX_PATH = str(Path("./colab_doc.leann").resolve())
builder = LeannBuilder(backend_name="hnsw")

for idx, line in enumerate(lines, start=1):
    builder.add_text(line, metadata={"line_number": idx})

builder.build_index(INDEX_PATH)
print(f"[LEANN] Successfully indexed {len(lines)} lines.")


KeyboardInterrupt: 

In [ ]:
# --- Step 2: Insert clean paragraph/window chunks into LEANN ---
import re

INDEX_PATH = str(Path("./colab_doc.leann").resolve())
MAX_WINDOW_LINES = 6   # max lines per chunk for long paragraphs
OVERLAP_LINES = 2      # overlap lines when splitting a long paragraph

try:
    builder = LeannBuilder(backend_name="hnsw", build_complexity=128)
except TypeError:
    builder = LeannBuilder(backend_name="hnsw")

# Group lines into paragraphs first (respecting blank lines or sentence-end -> uppercase line starts)
paragraphs: list[tuple[int, int, str]] = []  # (start_line_no, end_line_no, cleaned_text)
current_lines: list[tuple[int, str]] = []

for line_no, raw_line in enumerate(lines, start=1):
    cleaned_line = re.sub(r"-\s+$", "-", raw_line.strip())
    if not cleaned_line:
        if current_lines:
            p_text = re.sub(r"\s+", " ", " ".join(t for _, t in current_lines)).strip()
            paragraphs.append((current_lines[0][0], current_lines[-1][0], p_text))
            current_lines = []
        continue

    if current_lines:
        prev_text = current_lines[-1][1]
        if re.search(r"[.!?][\"')\]]?$", prev_text) and re.match(r"^[A-Z]", cleaned_line):
            p_text = re.sub(r"\s+", " ", " ".join(t for _, t in current_lines)).strip()
            paragraphs.append((current_lines[0][0], current_lines[-1][0], p_text))
            current_lines = [(line_no, cleaned_line)]
            continue

    current_lines.append((line_no, cleaned_line))

if current_lines:
    p_text = re.sub(r"\s+", " ", " ".join(t for _, t in current_lines)).strip()
    paragraphs.append((current_lines[0][0], current_lines[-1][0], p_text))

# Index short paragraphs whole; split very long paragraphs into overlapping line sub-windows
indexed_count = 0
for para_idx, (p_start, p_end, p_text) in enumerate(paragraphs, start=1):
    para_raw_lines = [
        re.sub(r"-\s+$", "-", lines[i - 1].strip())
        for i in range(p_start, p_end + 1)
        if lines[i - 1].strip()
    ]

    if len(para_raw_lines) <= MAX_WINDOW_LINES:
        builder.add_text(
            p_text,
            metadata={
                "paragraph_index": para_idx,
                "line_number": p_start,
                "line_start": p_start,
                "line_end": p_end,
            },
        )
        indexed_count += 1
    else:
        step = max(1, MAX_WINDOW_LINES - OVERLAP_LINES)
        for offset in range(0, len(para_raw_lines), step):
            sub_lines = para_raw_lines[offset : offset + MAX_WINDOW_LINES]
            if not sub_lines:
                break
            sub_text = re.sub(r"\s+", " ", " ".join(sub_lines)).strip()
            sub_start = p_start + offset
            sub_end = min(p_end, sub_start + len(sub_lines) - 1)
            builder.add_text(
                sub_text,
                metadata={
                    "paragraph_index": para_idx,
                    "line_number": sub_start,
                    "line_start": sub_start,
                    "line_end": sub_end,
                },
            )
            indexed_count += 1
            if sub_end >= p_end:
                break

builder.build_index(INDEX_PATH)
print(f"[LEANN] Successfully indexed {indexed_count} chunks from {len(lines)} lines.")

In [11]:
from leann import LeannSearcher

QUERIES = [
    "What did Darwin do between 1831 and 1836?",
]

searcher = LeannSearcher(INDEX_PATH)

for query in QUERIES:
    print("=" * 60)
    print(f"QUERY: {query}")

    try:
        results = searcher.search(query=query, top_k=5, complexity=64)
    except TypeError:
        results = searcher.search(query=query, top_k=5)

    seen_spans = set()
    for rank, item in enumerate(results, start=1):
        text = getattr(item, "text", None) or getattr(item, "content", None) or str(item)
        score = getattr(item, "score", None)
        metadata = getattr(item, "metadata", None) or {}

        span_key = (
            metadata.get("paragraph_index"),
            metadata.get("line_start"),
            metadata.get("line_end"),
            text,
        )
        if span_key in seen_spans:
            continue
        seen_spans.add(span_key)

        print(f"\n[{rank}] score={score} metadata={metadata}")
        print(text)

QUERY: What did Darwin do between 1831 and 1836?

[1] score=0.6627827882766724 metadata={'paragraph_index': 300, 'line_number': 2479, 'line_start': 2479, 'line_end': 2484}
Darwin returned home in 1836, and over the next twenty years he published a series of books which established his reputation as one of the leading biologists in England. As early as 1837, Darwin became convinced that animal and plant species were not fixed, but had evolved over the course of geologic history. At that time, however, he had no idea what might be the cause of such evolution. In 1838, however, he read An Essay on the Principle of Population by Thomas Malthus, and

[2] score=0.5616997480392456 metadata={'paragraph_index': 303, 'line_number': 2516, 'line_start': 2516, 'line_end': 2518}
skilled debater and a vigorous defender of Darwin's theories. The large majority of scientists had accepted the basic correctness of Darwin's theories by the time he died, in 1882.

[3] score=0.5517631769180298 metadata={'pa

In [6]:
import textwrap


def format_multiline_response(response, width: int = 88) -> str:
    if isinstance(response, dict):
        raw = response.get("response") or response.get("text") or response.get("content") or str(response)
    else:
        raw = (
            getattr(response, "response", None)
            or getattr(response, "text", None)
            or getattr(response, "content", None)
            or str(response)
        )

    # Convert literal "\n" sequences into real newlines
    text = str(raw).replace("\\n", "\n")

    formatted_lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            formatted_lines.append("")
            continue
        # Preserve bullet indentation while wrapping long lines cleanly
        if stripped.startswith(("- ", "* ")):
            formatted_lines.append(
                textwrap.fill(stripped, width=width, initial_indent="  ", subsequent_indent="    ")
            )
        else:
            formatted_lines.append(textwrap.fill(stripped, width=width))

    return "\n".join(formatted_lines)




In [20]:
from pathlib import Path
from leann import LeannChat

# --- Configure lightweight local/non-frontier model ---
llm_config = {
    "type": "hf",
    "model": "Qwen/Qwen3-0.6B",
    "temperature": 0.1,
    "max_new_tokens": 300,  # prevents abrupt cutoffs on multi-point answers
}

try:
    chat = LeannChat(INDEX_PATH, llm_config=llm_config,thinking_budget="low")
except TypeError:
    # Fallback if your installed leann version restricts llm_config keys
    chat = LeannChat(
        INDEX_PATH,
        llm_config=llm_config

    )

USER_QUERY = "What did Darwin do between 1831 and 1836?"

# --- Better-structured prompt for a lightweight (0.6B) model ---
# Explicitly asks for completeness across all retrieved chunks, strict grounding,
# and structured bullets so small models don't collapse everything into 1-2 lines.
prompt = f"""Answer the question below using ONLY the retrieved context from the index.
Do not guess or add outside facts not present in the context.
Read every retrieved chunk carefully and include all relevant nouns,events, numbers,dates, places, discoveries, observations, and notes that answer the question.

Format your answer with a brief summary sentence followed by bullet points so no details are missed.

Question: {USER_QUERY}"""

TOP_K = 4
SEARCH_COMPLEXITY = 128

try:
    leann_response = chat.ask(prompt, top_k=TOP_K, complexity=SEARCH_COMPLEXITY)
except TypeError:
    leann_response = chat.ask(prompt, top_k=TOP_K)

print(leann_response)
print(format_multiline_response(leann_response))



kwargs in HF:  {}
<think>

</think>

Between 1831 and 1836, Darwin observed many natural wonders, visited primitive tribes, discovered large numbers of fossils, and observed an enormous number of plant and animal species. He also took voluminous notes on everything he observed, which provided the basis for almost all his later work.
<think>

</think>

Between 1831 and 1836, Darwin observed many natural wonders, visited primitive tribes,
discovered large numbers of fossils, and observed an enormous number of plant and animal
species. He also took voluminous notes on everything he observed, which provided the
basis for almost all his later work.


In [ ]:
# --- Free GPU memory before loading Memorilla ---
del chat
del searcher
gc.collect()
torch.cuda.empty_cache()
print("[Memory] Cleared T4 VRAM for Memorilla.")

In [ ]:
!git clone https://github.com/snap-stanford/memorilla.git
%cd memorilla
!pip install -q --no-deps -e .
!pip install -q torch transformers accelerate bitsandbytes huggingface_hub safetensors einops



In [ ]:
import inspect
import torch
from huggingface_hub import snapshot_download
from memorilla import MemoryModule

ckpt_dir = snapshot_download(
    repo_id="memorilla/Memorilla-Qwen3-8B",
    allow_patterns=["stage3/*"]
)

# Load the 194M memory module in float16 (T4 does not natively accelerate bfloat16)
memory = MemoryModule.from_pretrained(f"{ckpt_dir}/stage3").to("cuda", dtype=torch.float16).eval()

# Print the forward signature so you can confirm exact parameter names if needed
print("MemoryModule forward signature:", inspect.signature(memory.forward))


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

MemoryModule forward signature: (doc_embeds: torch.Tensor, doc_padding_mask: torch.Tensor | None, question_embeds: torch.Tensor) -> torch.Tensor


In [ ]:
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 1. Load Document & Query Encoder (4-bit)
enc_id = "Qwen/Qwen3-Embedding-4B"
enc_tokenizer = AutoTokenizer.from_pretrained(enc_id, padding_side="left")
encoder = AutoModel.from_pretrained(
    enc_id,
    quantization_config=bnb_config,
    device_map="auto"
).eval()

# 2. Load Frozen Base LLM (4-bit)
llm_id = "Qwen/Qwen3-8B"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_id)
llm = AutoModelForCausalLM.from_pretrained(
    llm_id,
    quantization_config=bnb_config,
    device_map="auto"
).eval()

print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (12 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

VRAM allocated: 9.15 GB


In [ ]:
# --- 1. Helper to Read and Split Your Text File into Chunks ---
def load_and_chunk_txt(file_path: str, chunk_size: int = 1200, overlap: int = 150) -> list[str]:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read().strip()

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# --- 2. Batched Embedding Function (Prevents T4 Out-Of-Memory Errors) ---
@torch.no_grad()
def get_qwen3_embeddings(texts: list[str], batch_size: int = 16, max_length: int = 512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        batch = enc_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to("cuda")

        outputs = encoder(**batch)
        last_token_indices = batch.attention_mask.sum(dim=1) - 1
        embeddings = outputs.last_hidden_state[
            torch.arange(len(batch_texts), device="cuda"), last_token_indices
        ]
        norm_embeds = torch.nn.functional.normalize(embeddings, p=2, dim=-1).to(torch.float16)
        all_embeddings.append(norm_embeds)

    return torch.cat(all_embeddings, dim=0)

# --- 3. Encode Your File Once and Query Multiple Times ---
FILE_PATH = "/content/LeanNInput.txt"  # <-- Change this to your uploaded filename

docs = load_and_chunk_txt(FILE_PATH)
print(f"Loaded {len(docs)} chunks from {FILE_PATH}")

# Pre-compute document embeddings once so you can ask multiple questions quickly
cached_doc_embeds = get_qwen3_embeddings(docs).unsqueeze(0)

@torch.no_grad()
def ask_file(question: str, max_new_tokens: int = 150):
    # Encode only the question
    query_embed = get_qwen3_embeddings([question])

    # Compress all file chunks into 16 query-dependent latent memory tokens
    memory_tokens = memory(cached_doc_embeds, query_embed)

    # Format prompt and convert to embeddings
    messages = [{"role": "user", "content": question}]
    prompt_text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    input_ids = llm_tokenizer(prompt_text, return_tensors="pt").input_ids.to("cuda")
    text_embeds = llm.get_input_embeddings()(input_ids).to(torch.float16)

    # Prepend the 16 latent memory vectors to the prompt
    combined_embeds = torch.cat([memory_tokens.to(text_embeds.dtype), text_embeds], dim=1)
    attention_mask = torch.ones((1, combined_embeds.shape[1]), dtype=torch.long, device="cuda")

    # Generate answer
    output_ids = llm.generate(
        inputs_embeds=combined_embeds,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )
    return llm_tokenizer.decode(output_ids[0], skip_special_tokens=True)

# --- 4. Ask Questions About Your File ---
question = "What did Darwin do between 1831 and 1836?."
print("Answer:", ask_file(question))


Loaded 893 chunks from /content/LeanNInput.txt


TypeError: MemoryModule.forward() missing 1 required positional argument: 'question_embeds'

In [ ]:
# --- 4. Ask Questions About Your File ---
question = "What did Darwin do between 1831 and 1836?."
print("Answer:", ask_file(question))